<a href="https://colab.research.google.com/github/rajeevgargsg/EssayEval/blob/main/18MarEssay_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- SETUP CELL ---
import sys

# Install/Update core libraries with specific versions to prevent the dependency error
!pip install -U langchain-groq langgraph tavily-python requests==2.32.5 --quiet

# Restart the session automatically to apply the correct 'requests' version
if 'google.colab' in sys.modules:
    import IPython
    print("Setup complete. Restarting session...")
    IPython.Application.instance().kernel.do_shutdown(True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
Setup complete. Restarting session...


In [3]:
import os
from typing import List, TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
from tavily import TavilyClient
from google.colab import userdata

# 1. API Setup
os.environ["GROQ_API_KEY"] = userdata.get('GroqRGAPI01')
os.environ["TAVILY_API_KEY"] = userdata.get('TavilyRGAPI01')

# 2026 Production Models
primary_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
judge_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1)
tavily = TavilyClient()

# 2. State Definition
class EssayState(TypedDict):
    topic: str
    target_word_count: int
    selected_queries: List[str]
    retrieved_content: str
    source_links: List[str]
    first_draft: str
    review_feedback: str
    correction_count: int
    final_essay: str
    quality_score: str

# 3. Graph Nodes

def query_generator(state: EssayState):
    """Generates queries for the March 2026 conflict status."""
    prompt = f"""Generate 6 search queries for the CURRENT March 2026 status of: {state['topic']}.
    Focus on:
    1. Operation Epic Fury developments.
    2. Status of the Strait of Hormuz and oil prices.
    3. Leadership changes in Tehran (Mojtaba Khamenei).
    4. Diplomatic reactions from the GCC and China."""
    res = primary_llm.invoke(prompt).content
    queries = [line.strip().strip('"') for line in res.split('\n') if any(char.isdigit() for char in line[:2])]
    return {"selected_queries": queries[:6]}

def web_researcher(state: EssayState):
    """Gathers live 2026 intelligence."""
    context = ""
    urls = []
    print(f"Gathering intel on: {state['selected_queries']}")
    for query in state['selected_queries']:
        try:
            search = tavily.search(query=query, search_depth="advanced", max_results=2)
            for r in search["results"]:
                urls.append(r['url'])
                context += f"\nSOURCE: {r['url']}\nCONTENT: {r['content']}\n"
        except: continue
    return {"retrieved_content": context, "source_links": urls}

def drafter(state: EssayState):
    prompt = f"""Write a comprehensive 1,200-word geopolitical essay on: {state['topic']}.
    Current Date: March 18, 2026.

    Use this research:
    {state['retrieved_content']}

    Structure:
    - Introduction: The shift from shadow war to direct conflict (Feb 28).
    - Military Analysis: Impact of 'Operation Epic Fury' and Iranian retaliatory strikes.
    - Economic Fallout: The Strait of Hormuz blockade and global energy shock.
    - Leadership: The transition to Mojtaba Khamenei.
    - Conclusion: Strategic outlook for the remainder of 2026."""
    draft = primary_llm.invoke(prompt).content
    return {"first_draft": draft}

def reviewer(state: EssayState):
    """Checks for bias and factual consistency with 2026 reports."""
    prompt = f"Review this essay for geopolitical balance and factual consistency with March 2026 events: {state['first_draft']}. List errors or biased phrases with a '-'."
    feedback = primary_llm.invoke(feedback_prompt := prompt).content
    count = feedback.count("\n-")
    return {"review_feedback": feedback, "correction_count": count}

def writer(state: EssayState):
    prompt = f"Refine the essay based on this feedback: {state['review_feedback']}. Ensure a professional, analytical tone."
    final = primary_llm.invoke(prompt).content
    return {"final_essay": final}

def quality_judge(state: EssayState):
    prompt = f"Rate this essay (1-10) for analytical depth regarding the 2026 US-Iran War.\n\n{state['final_essay']}"
    return {"quality_score": judge_llm.invoke(prompt).content}

# 4. Compile Graph
builder = StateGraph(EssayState)
builder.add_node("planner", query_generator)
builder.add_node("researcher", web_researcher)
builder.add_node("drafter", drafter)
builder.add_node("reviewer", reviewer)
builder.add_node("writer", writer)
builder.add_node("judge", quality_judge)

builder.set_entry_point("planner")
builder.add_edge("planner", "researcher")
builder.add_edge("researcher", "drafter")
builder.add_edge("drafter", "reviewer")
builder.add_edge("reviewer", "writer")
builder.add_edge("writer", "judge")
builder.add_edge("judge", END)

graph = builder.compile()

# 5. Run
initial_input = {
    "topic": "USA & Iran Relationship and Current 2026 Conflict",
    "target_word_count": 1200
}

result = graph.invoke(initial_input)

print("\n" + "="*60)
print("GEOPOLITICAL INTELLIGENCE REPORT: MARCH 2026")
print("="*60)
print(f"Sources Verified: {len(result['source_links'])}")
print(f"Critique Adjustments: {result['correction_count']}")
print(f"Final Audit: {result['quality_score']}")
print("-" * 60)
print(result['final_essay'])

Gathering intel on: ['1. "Operation Epic Fury update March 2026" - This search query aims to find the latest developments on the operation, including any recent military actions, casualties, or changes in strategy.', '2. "Strait of Hormuz blockade 2026 oil price impact" - This query focuses on the current status of the Strait of Hormuz, a critical waterway for global oil trade, and how the conflict is affecting oil prices and the global economy.', '3. "Mojtaba Khamenei Iran leadership changes 2026" - This search query seeks to find information on the current leadership situation in Tehran, particularly regarding Mojtaba Khamenei, and how any changes may be impacting the country\'s relations with the USA and the ongoing conflict.', '4. "GCC reaction to US-Iran conflict 2026" - This query looks for the latest diplomatic reactions from the Gulf Cooperation Council (GCC) countries, such as Saudi Arabia, the UAE, and Qatar, to the ongoing conflict between the USA and Iran.', '5. "China\'s s